### Analysis Bilevel

Main text run date (Revision 1): <b>October 19, 2025</b>

DD+MibS and DD+BilevelSolver (Revision 2): <b>March 24, 2026</b>

In [ ]:
# Results date
# run_date = "Sep24_2025"
# run_date = "Oct8_2025"
# run_date = "Oct14_2025_cont"
run_date = "Oct19_2025_cont"

In [ ]:
# Packages
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

# Disable SettingWithCopyWarning
pd.options.mode.chained_assignment = None  # default='warn'


In [ ]:
# Create folder 'figures' if it does not exist
import os
if not os.path.exists("figures"):
    os.makedirs("figures")

### Styles and Style Mappings

In [ ]:
sns.set_theme(style="whitegrid")
fontsize = 12

# Colors
colors={"dd": "#1f77b4", "bilevelsolver": "#ff7f0e", "mibs": "#2ca02c"}

# Markers
markers={"dd": "o", "bilevelsolver": "s", "mibs": "D"}

# Labels
labels={"dd": "DD", "bilevelsolver": "B&C", "mibs": "MibS"}

# Line styles
linestyles = {
    "dd": "solid",             
    "bilevelsolver": "dashed", 
    "mibs": "dotted"
}

# Line width
linewidth=2.25


### Data

In [ ]:
# Collect information about instances
df_info = pd.read_csv("collection-benchmark.csv")
print("Number of instances downloadable from BOBILib (full collection):", len(df_info))
df_info.head(3)

In [ ]:
# Collect information about tested benchmark instances (openset_500)
df_info_benchmark = pd.read_csv("benchmark_info.csv")
print("Number of instances tested (openset_500):", len(df_info_benchmark))
df_info_benchmark.head(3)

In [ ]:
# Set instance class
df_info_benchmark.loc[ 
    df_info_benchmark["name"].str.contains("interKP"),
    ["class", "class2"] 
] = ["interKP", "knapsack"]

df_info_benchmark.loc[ 
    df_info_benchmark["name"].str.contains("OR"),
    ["class", "class2"]
] = ["OR", "multidimensional knapsack"]

df_info_benchmark.loc[ 
    df_info_benchmark["name"].str.contains("weish"),
    ["class", "class2"]
] = ["weish", "multidimensional knapsack"]


df_info_benchmark.loc[ 
    (df_info_benchmark["name"].str.contains("general")),
    ["class", "class2"]
] = ["general", "general"]

# df_info_benchmark.loc[ 
#     df_info_benchmark["name"].str.contains("sento"),
#     ["class", "class2"]
# ] = ["sento", "multidimensional knapsack"]

df_info_benchmark.loc[ 
    df_info_benchmark["name"].str.contains("K50"),
    ["class", "class2"]
] = ["K50", "knapsack"]

df_info_benchmark.loc[
    (df_info_benchmark["class"].isna()) & (df_info_benchmark["type"] == "general-bilevel"),
    ["class", "class2"]
] = ["others/general", "general"]

df_info_benchmark.loc[
    (df_info_benchmark["class"].isna()) & (df_info_benchmark["type"] == "interdiction"),
    ["class", "class2"]
] = ["others/interdiction", "general"]


df_info_benchmark["class"] = df_info_benchmark["class"].fillna("other")
df_info_benchmark["class2"] = df_info_benchmark["class2"].fillna("multiple")

### Table 1 - Distribution of testbed BOBILib instances

In [ ]:
# Create table with number of instances per class
df_count = df_info_benchmark.groupby(["class", "type", "class2"]).size().reset_index(name='counts')
df_count.sort_values(by="counts", ascending=False)

In [ ]:
import pandas as pd
import numpy as np

# --- Configuration ---
use_booktabs = True  # set to False if you will not \usepackage{booktabs}
sort_by = "Count"
descending = True

# --- Group and aggregate directly from df_info_benchmark ---
gb_cols = ["class", "type", "class2"]
leader_col = "upper_variables"
follower_col = "lower_variables"
state_col = "state_variables"   # new column

# Count: prefer unique instance id if available; fall back to group size
if "name" in df_info_benchmark.columns:
    counts = df_info_benchmark.groupby(gb_cols)["name"].count()
else:
    counts = df_info_benchmark.groupby(gb_cols).size().rename("name")

agg_minmax = (
    df_info_benchmark
    .groupby(gb_cols)
    .agg(
        leader_min=(leader_col, "min"),
        leader_max=(leader_col, "max"),
        follower_min=(follower_col, "min"),
        follower_max=(follower_col, "max"),
        state_min=(state_col, "min"),
        state_max=(state_col, "max"),
    )
)

summary = (
    pd.concat([counts.rename("Count"), agg_minmax], axis=1)
      .reset_index()
)

# Format ranges
def fmt_range(a, b):
    if pd.isna(a) or pd.isna(b):
        return ""
    return f"{int(np.floor(a))}--{int(np.floor(b))}"

summary["Leader vars"]   = summary.apply(lambda r: fmt_range(r["leader_min"], r["leader_max"]), axis=1)
summary["Follower vars"] = summary.apply(lambda r: fmt_range(r["follower_min"], r["follower_max"]), axis=1)
summary["State vars"]    = summary.apply(lambda r: fmt_range(r["state_min"], r["state_max"]), axis=1)

# Select and rename display columns
out = (
    summary[["class", "Count", "type", "class2", "Leader vars", "Follower vars", "State vars"]]
      .rename(columns={"class": "Class", "type": "Type", "class2": "Subclass"})
      .sort_values(by=sort_by, ascending=not descending)
      .reset_index(drop=True)
)

# --- Build LaTeX manually ---
cols = ["Class", "Count", "Subclass", "Leader vars", "Follower vars", "State vars"]

if use_booktabs:
    header = "\\begin{table}[h!]\n\\centering\n\\begin{tabular}{lllllll}\n\\toprule\n"
    mid    = "\\midrule\n"
    bottom = "\\bottomrule\n\\end{tabular}\n"
else:
    header = "\\begin{table}[h!]\n\\centering\n\\begin{tabular}{lllllll}\n\\hline\n"
    mid    = "\\hline\n"
    bottom = "\\hline\n\\end{tabular}\n"

latex = header
latex += " {} \\\\\n".format(" & ".join(cols))
latex += mid

for _, row in out.iterrows():
    latex += " {} \\\\\n".format(" & ".join(str(row[c]) for c in cols))

latex += bottom
latex += "\\caption{Benchmark instances by class, with ranges of leader, follower, and state variables.}\n"
latex += "\\label{tab:instances_ranges}\n"
latex += "\\end{table}\n"

print(latex)

In [ ]:
# Read results files
df = pd.read_csv(f"results_{run_date}.csv")
df.head(3)


In [ ]:
# Check number of instances reported by each solver
for solver in ["bilevelsolver", "mibs"]:
    n_instances = df[df["solver"] == solver]["name"].nunique()
    print(f"Solver {solver}: {n_instances} instances")

# DD solver also should consider width
df_dd = df[df["solver"] == "dd"]
for width in df_dd["width"].unique():
    n_instances = df_dd[df_dd["width"] == width]["name"].nunique()
    print(f"Solver dd with width {width}: {n_instances} instances")

In [ ]:
# Check instances that are in df_info_benchmark but not in results
result_instances = df["name"].unique()
info_instances = df_info_benchmark["name"].unique()
missing_instances = set(info_instances) - set(result_instances)
print(f"Instances in df_info_benchmark but not in results: {missing_instances}")


In [ ]:
# Incorporate the column class, class2 from df_info_benchmark into df
df = df.merge(
    df_info_benchmark[["name", "class", "class2", "lower_continuous_variables", "lower_integer_variables", "has_noninteger_coefficients"]],
    on="name",
    how="left"
)
df.head(3)

### General Parameters

- Feasible solutions: NaN in the filed "bestobj"
- Optimality Threshold: 1e-10
- Time limit: 7200

In [ ]:
# If it has feasible solution
df["feasible"] = ~df["bestobj"].isna()

# If improved best known bounds from the BOBILib
df["improved_known_bobilib"] = (df["feasible"] & df["known_objective_value"].isna()) | (df["bestobj"] <= df["known_objective_value"]-0.1)


In [ ]:
# Print instances with numerical issues due to large objective values (see discussion in Section 6)
# We marked them with negative gaps
df_numerical_issues = df[ (df["solver"] == "dd") & (df["optgap"] < 0) ][["name","width","bestobj", "bestbound", "optgap"]]
df_numerical_issues

In [ ]:
# Collect names
instance_names = list(df_numerical_issues["name"].unique())
instance_names

In [ ]:
df.columns

In [ ]:
# For numerical consistency, we will exclude those instances (marked as 'unsolved');
df = df[~((df["solver"] == "dd") & (df["optgap"] < 0))]

In [ ]:
# We will also generally exclude harp2-0.900000 from DD, which suffers of similar numerical issues
df = df[~((df["solver"] == "dd") & (df["name"] == "harp2-0.900000"))]

In [ ]:
# If instance is solved to optimality
opt_threshold = 1e-10
df["solved"] = (df["optgap"].abs() < opt_threshold) & df["feasible"]

In [ ]:
timelimit = 7200

In [ ]:
df.columns

### Cleaning & Setup

Check if the optimal solution values are matching

In [ ]:
# Check if optimal solutions match (for all widths of DD)
dfDD = df[ (df["solver"] == "dd") & (df["solved"] == True) ]
dfBilevel = df[ (df["solver"] == "bilevelsolver") & (df["solved"] == True) ]

df_check = pd.merge(
    dfDD,
    dfBilevel,
    on="name",
    suffixes=("_dd", "_bilevel")
)

df_check["obj_match"] = df_check["bestobj_dd"] == df_check["bestobj_bilevel"]
print(f"Number of instances with matching optimal solutions: {df_check['obj_match'].sum()} out of {len(df_check)}")

In [ ]:
# Instance solved by BC but not by DD (width 8)
dfBC = df[ (df["solver"] == "bilevelsolver") & (df["solved"] == True) ]
dfDD_not_solved = df[ (df["solver"] == "dd") & (df["solved"] == False) & (df["width"] == 8) ]
df_check_BC_not_DD = pd.merge(
    dfBC,
    dfDD_not_solved,
    on="name",
    suffixes=("_bc", "_dd")
)
df_check_BC_not_DD

In [ ]:
# What was the DD gap on these instances?
df_check_BC_not_DD["optgap_dd"].describe()

In [ ]:
# Check if optimal solutions match
dfDD = df[ (df["solver"] == "dd") & (df["solved"] == True) ]
dfMibs = df[ (df["solver"] == "mibs") & (df["solved"] == True) ]

df_check = pd.merge(
    dfDD,
    dfMibs,
    on="name",
    suffixes=("_dd", "_mibs")
)

df_check["obj_match"] = df_check["bestobj_dd"] == df_check["bestobj_mibs"]
print(f"Number of instances with matching optimal solutions: {df_check['obj_match'].sum()} out of {len(df_check)}")

In [ ]:
# Check number of processed nodes for MibS and B&C. If "0," it should not have a meaningful gap or root bound
df[ ( (df["solver"] == "mibs") | (df["solver"] == "bilevelsolver") ) & (df["nodes"] == 0) ] [["optgap", "rootbound", "bestbound"]]

If number of processed nodes are 0 for MibS or B&C, optimality gap or root bound is not considered

In [ ]:
df.loc[ 
    ( (df["solver"] == "mibs") | (df["solver"] == "bilevelsolver") ) & (df["nodes"] == 0),
    ["optgap", "rootbound", "bestbound"]
] = [np.nan, np.nan, np.nan]

df[ ((df["solver"] == "mibs") | (df["solver"] == "bilevelsolver")) & (df["nodes"] == 0) ]

In [ ]:
# Calculate best known objective values across all methods
df['min_bestobj'] = df.groupby('name')['bestobj'].transform('min')

# Calculate primal gap, i.e., distance to best known objective
df['primal_gap'] = df.apply(
    lambda row: (row['bestobj'] - row['min_bestobj']) / abs(row['min_bestobj']) if row['feasible'] else np.nan,
    axis=1
)

# Calculate MDD root gap
df["mddrootgap"] = df.apply(
    lambda row: (row['min_bestobj'] - row['mddrootbound']) / abs(row['min_bestobj']) if row['feasible'] else np.nan,
    axis=1
)

# Calculate root gap
df["rootgap"] = df.apply(
    lambda row: (row['min_bestobj'] - row['rootbound']) / abs(row['min_bestobj']) if row['feasible'] else np.nan,
    axis=1
)

# # Calculate root gap
# df["hprgap"] = df.apply(
#     lambda row: (row['min_bestobj'] - row['hprbound']) / abs(row['min_bestobj']) if row['feasible'] else np.nan,
#     axis=1
# )

In [ ]:
df[["name", "solver", "bestobj", "primal_gap", "min_bestobj"]].head(10)

In [ ]:
df[ (df["feasible"])].sort_values(by="state_variables", ascending=False)[["name", "state_variables"]]

<b>Base MDD width for comparisons</b>


In [ ]:
base_width = 8

<b>General Statistics</b>

Main methods: <i>bilevelsolver</i>, <i>mibs</i>, <i>dd</i>

Note: <i>dd_nologicalcuts</i> and <i>mibs_general</i> were dominated in total performance but will be analyzed later

In [ ]:
# Merge dataframes: only consider base width for DD
dfDD = df[ (df["solver"] == "dd") & (df["width"] == base_width) ]
dfMibs = df[df["solver"] == "mibs"]
dfBilevel = df[df["solver"] == "bilevelsolver"]

dfDD_pref = (
    dfDD
    .rename(columns={
        "bestobj"                : "dd_bestobj",
        "time"                   : "dd_time",
        "optgap"                 : "dd_optgap",
        "solved"                 : "dd_solver_solved",
        "benchmark"              : "benchmark",
        "feasible"               : "dd_feasible",
        "improved_known_bobilib" : "dd_improved_known_bobilib",
        "primal_gap"             : "dd_primal_gap",
    })
    .drop(columns=["solver"])
)

dfBilevel_pref = (
    dfBilevel
    .rename(columns={
        "bestobj"                : "bilevelsolver_bestobj",
        "time"                   : "bilevelsolver_time",
        "optgap"                 : "bilevelsolver_optgap",
        "solved"                 : "bilevelsolver_solver_solved",
        "feasible"               : "bilevelsolver_feasible",
        "improved_known_bobilib" : "bilevelsolver_improved_known_bobilib",
        "primal_gap"             : "bilevelsolver_primal_gap",
    })
    .drop(columns=["solver"])
)

dfMibs_pref = (
    dfMibs
    .rename(columns={
        "bestobj"                    : "mibs_bestobj",
        "time"                       : "mibs_time",
        "optgap"                     : "mibs_optgap",
        "solved"                     : "mibs_solver_solved",
        "feasible"                   : "mibs_feasible",
        "improved_known_bobilib"     : "mibs_improved_known_bobilib",
        "primal_gap"                 : "mibs_primal_gap",
    })
    .drop(columns=["solver"])
)

# Merge & validate one-to-one relationship
key_cols = ["name", "type", "upper_variables", "lower_variables", "upper_binary_variables", "known_objective_value", "status", "state_variables", "class", "class2"]
df_aggregated = (
    dfDD_pref
    .merge(dfBilevel_pref, on=key_cols, how="outer", validate="one_to_one")
    .merge(dfMibs_pref,   on=key_cols, how="outer", validate="one_to_one")
)

# Empty feasible indicates no feasible solution found
df_aggregated["dd_feasible"] = df_aggregated["dd_feasible"].astype("boolean").fillna(False)
df_aggregated["bilevelsolver_feasible"] = df_aggregated["bilevelsolver_feasible"].astype("boolean").fillna(False)
df_aggregated["mibs_feasible"] = df_aggregated["mibs_feasible"].astype("boolean").fillna(False)

df_aggregated["dd_solver_solved"] = df_aggregated["dd_solver_solved"].astype("boolean").fillna(False)
df_aggregated["bilevelsolver_solver_solved"] = df_aggregated["bilevelsolver_solver_solved"].astype("boolean").fillna(False)
df_aggregated["mibs_solver_solved"] = df_aggregated["mibs_solver_solved"].astype("boolean").fillna(False)

columns = [col for col in df_aggregated.columns.tolist() if "dd" in col or "bilevelsolver" in col or "mibs" in col or "name" in col or "width" in col or "class" in col or "class2" in col or "type" in col or "known_objective_value" in col or "state" in col or ""]

df_aggregated[columns]


In [ ]:
solvers = ["dd", "bilevelsolver", "mibs"]
df_solvers = df[df["solver"].isin(solvers)]

num_instances = df_solvers["name"].nunique()
print("Number of instances reported in any of the solvers:", num_instances)

for solver, df_solver in df_solvers.groupby("solver"):
    print("\nSolver:", solver)
    if solver == "dd":
        for width, df_width in df_solver.groupby("width"):
            print(f"\tWidth {width}:")
            print("\t\tProved optimality:", df_width["solved"].sum())
            print("\t\tImproved known (from website):", df_width["improved_known_bobilib"].sum())
            print("\t\tFeasible:", df_width["feasible"].sum())
            print("\t\tMemory limit or error:", num_instances - df_width.shape[0])
    else:
        print("\tProved optimality:", df_solver["solved"].sum())
        print("\tImproved known (from website):", df_solver["improved_known_bobilib"].sum())
        print("\tFeasible:", df_solver["feasible"].sum())
        print("\tMemory limit or error:", num_instances - df_solver.shape[0])

### Table 2: Summary of solver results

Note: The manuscript reports results obtained with the submission version of the code. After submission, we identified a small implementation refinement that slightly improves feasibility detection for DD, specifically by increasing the number of checks performed before the time limit is reached. The replication package includes this updated code, and therefore a very small number of feasibility counts may be slightly larger than those reported in the paper. The tables in the manuscript were left unchanged to preserve consistency with the version evaluated during review.

In [ ]:
# Collect solvers
dfDD = df[ (df["solver"] == "dd") & (df["width"] == 8) ]
dfBilevel = df[ (df["solver"] == "bilevelsolver") ]
dfMibs = df[ (df["solver"] == "mibs") ]

summary_results = {
    "Method": [],
    "Proved Optimality": [],
    "Improved Best Known": [],
    "Has Feasible Solution": [],
    "Better DD": [],
    "Better Bilevel": [],
    "Better Mibs": [],
    "Better All": [],
    "Avg Gap": [],
    "Std Gap": [],
    "Primal Gap": [],
    "% Gap <= 1%": [],
    "% Gap <= 10%": [],
    "% Gap <= 100%": [],
}

for method, df_method in [("dd", dfDD), ("bilevelsolver", dfBilevel), ("mibs", dfMibs)]:
    # General stats
    summary_results["Method"].append(method)
    summary_results["Proved Optimality"].append(int(df_method["solved"].sum()))
    summary_results["Improved Best Known"].append(int(df_method["improved_known_bobilib"].sum()))
    summary_results["Has Feasible Solution"].append(int(df_method["feasible"].sum()))
    summary_results["Avg Gap"].append(
        round(100*df_method[ (df_method["feasible"]) & (~df_method["solved"]) ]["optgap"].mean(), 2)
    )
    summary_results["Std Gap"].append(
        round(100*df_method[ (df_method["feasible"]) & (~df_method["solved"]) ]["optgap"].std(), 2)
    )
    summary_results["Primal Gap"].append(
        round(100*df_method[ df_method["feasible"] ]["primal_gap"].mean(), 2)
    )
    summary_results["% Gap <= 1%"].append(
        round(
            100* (df_method[ (df_method["feasible"]) & (df_method["optgap"] <= 0.01)].shape[0])/(df_method[ (df_method["feasible"])].shape[0])
        ,2)
    )
    summary_results["% Gap <= 10%"].append(
        round(
            100* (df_method[ (df_method["feasible"]) & (df_method["optgap"] <= 0.1)].shape[0])/(df_method[ (df_method["feasible"])].shape[0])
        ,2)
    )
    summary_results["% Gap <= 100%"].append(
        round(
            100* (df_method[ (df_method["feasible"]) & (df_method["optgap"] <= 1)].shape[0])/(df_method[ (df_method["feasible"])].shape[0])
        ,2)
    )
    
    # Comparison statistics
    summary_results["Better DD"].append(
        int( 
            ( (df_aggregated[f"{method}_feasible"]) & ( (~df_aggregated["dd_feasible"]) | (df_aggregated[f"{method}_bestobj"] <= df_aggregated["dd_bestobj"]-0.1) ) ).sum()  
        ) if method != "dd" else np.nan
    )

    summary_results["Better Bilevel"].append(
        int( 
            ((df_aggregated[f"{method}_feasible"]) & ( (~df_aggregated["bilevelsolver_feasible"]) | (df_aggregated[f"{method}_bestobj"] <= df_aggregated["bilevelsolver_bestobj"]-0.1) ) ).sum()  
        ) if method != "bilevelsolver" else np.nan
    )

    summary_results["Better Mibs"].append(
        int( 
            ((df_aggregated[f"{method}_feasible"]) & ( (~df_aggregated["mibs_feasible"]) | (df_aggregated[f"{method}_bestobj"] <= df_aggregated["mibs_bestobj"]-0.1) ) ).sum()  
        ) if method != "mibs" else np.nan
    )


    # Better than all others
    others = [m for m in ["dd", "bilevelsolver", "mibs"] if m != method]
    cond = df_aggregated[f"{method}_feasible"] & df_aggregated[f"{method}_improved_known_bobilib"]
    for o in others:
        cond = cond & ( (~df_aggregated[f"{o}_feasible"]) |
                        (df_aggregated[f"{method}_bestobj"] < df_aggregated[f"{o}_bestobj"]) )
    summary_results["Better All"].append(int(cond.sum()))

df_summary = pd.DataFrame(summary_results)
# df_summary["Better Bilevel"]
# print("\nSummary of Results:")
df_summary

<b>Summary Table</b>

In [ ]:
import pandas as pd
import numpy as np

# ---------- Settings ----------
use_booktabs = True
caption = "Summary of solver results on the BOBILib testbed."
label   = "tab:summary_results"

# Row order and LaTeX names
method_order = ['mibs', 'bilevelsolver', 'dd']
method_latex = {'mibs': r'$\MibS{}$', 'bilevelsolver': r'$\BC{}$', 'dd': r'$\DD{}$'}

# Pairwise header strings (match your table)
col_lt_mibs = r"$\;\;< z_{\MibS{}}\;\;$"
col_lt_bc   = r"$\;\;< z_{\BC{}}\;\;$"
col_lt_dd   = r"$\;\;< z_{\DD{}}\;\;$"

dfT = df_summary.copy()

# ----- Assemble output with exact headers -----
df_out = pd.DataFrame({
    "Method": dfT["Method"],
    "Feasible": dfT["Has Feasible Solution"],
    "Optimal": dfT["Proved Optimality"],
    "Improved Best": dfT["Better All"],            # <- from Better All
    r"% Gap $\le$ 1\%": dfT["% Gap <= 1%"],
    r"% Gap $\le$ 10\%": dfT["% Gap <= 10%"],
    r"% Gap $\le$ 100\%": dfT["% Gap <= 100%"],
    col_lt_mibs: dfT["Better Mibs"],
    col_lt_bc:   dfT["Better Bilevel"],
    col_lt_dd:   dfT["Better DD"],
})

# Map method names and enforce order MibS -> BC -> DD
df_out["Method"] = df_out["Method"].map(method_latex).fillna(df_out["Method"])
order_key = df_summary["Method"].map({k: i for i, k in enumerate(method_order)})
if order_key.isna().all():  # fallback if df_summary already mapped
    order_key = df_out["Method"].map({method_latex[k]: i for i, k in enumerate(method_order)})
df_out = df_out.assign(_ord=order_key).sort_values("_ord").drop(columns="_ord").reset_index(drop=True)

# Format integers with thousands separators for counts and pairwise wins
count_cols = ["Feasible", "Optimal", "Improved Best", col_lt_mibs, col_lt_bc, col_lt_dd]
for c in count_cols:
    df_out[c] = df_out[c].apply(lambda x: f"{int(x):,}" if pd.notna(x) else "--")

# Format percentage columns with one decimal and a percent sign
pct_cols = [r"% Gap $\le$ 1\%", r"% Gap $\le$ 10\%", r"% Gap $\le$ 100\%"]
for c in pct_cols:
    df_out[c] = df_out[c].apply(lambda x: f"{float(x):.1f}\\%" if pd.notna(x) else "--")

# ----- Build LaTeX -----
cols = ["Method", "Feasible", "Optimal", "Improved Best",
        r"% Gap $\le$ 1\%", r"% Gap $\le$ 10\%", r"% Gap $\le$ 100\%",
        col_lt_mibs, col_lt_bc, col_lt_dd]

align_spec = "l" + "r"*(len(cols)-1)
if use_booktabs:
    head = (f"\\begin{{table}}[h!]\n\\centering\n\\small\n\\caption{{{caption}}}\n"
            f"\\label{{{label}}}\n\\begin{{tabular}}{{{align_spec}}}\n\\toprule\n")
    mid  = "\\midrule\n"
    foot = "\\bottomrule\n\\end{tabular}\n\\end{table}\n"
else:
    head = (f"\\begin{{table}}[h!]\n\\centering\n\\small\n\\caption{{{caption}}}\n"
            f"\\label{{{label}}}\n\\begin{{tabular}}{{{align_spec}}}\n\\hline\n")
    mid  = "\\hline\n"
    foot = "\\hline\n\\end{tabular}\n\\end{table}\n"

latex = head
latex += " {} \\\\\n".format(" & ".join(cols))
latex += mid
for _, row in df_out.iterrows():
    latex += " {} \\\\\n".format(" & ".join(str(row[c]) for c in cols))
latex += foot

print(latex)




<b>Cumulative time plot</b>

In [ ]:
gap_thresholds = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0]
gap_labels = [f"{int(g*100)}%" for g in gap_thresholds]
gap_marks = [timelimit + 360 * (i+1) for i in range(len(gap_thresholds))]
gap_map = {g: m for g, m in zip(gap_thresholds, gap_marks)}

for solver in solvers:
    d = df[ (df["solver"] == solver) ].copy()
    if solver == "dd":
        d = d[ d["width"] == base_width ]
    # Instances solved optimally
    solved_times = d.loc[d["solved"], "time"].clip(upper=timelimit).values
    solved_times = np.sort(solved_times)
    x = list(solved_times)
    y = list(np.arange(1, len(solved_times) + 1))
    
    # Extend with gap thresholds
    for i, g in enumerate(gap_thresholds):
        count = (d["optgap"] <= g).sum()
        x.append(gap_marks[i])  # artificial spacing beyond timelimit
        y.append(count)
    
    plt.step(x, y, where="post", label=labels[solver], color=colors[solver], linestyle=linestyles[solver], lw=linewidth)

# # x_ticks = [0, 1800, 3600, 5400, 7200] + gap_marks
# # x_ticks_labels = ["0", "1800", "3600", "5,400", "7,200"] + [f"{g:.0%}" for g in gap_thresholds]

gaps_to_plot = [0.25, 0.5, 0.75, 1]
x_ticks = [0, 1800, 3600, 5400, 7200] + [gap_map[g] for g in gaps_to_plot]
# x_ticks_labels = ["0", "1,800", "3,600", "5,400", "7,200"] + [f"{g:.0%}" for g in gaps_to_plot]
x_ticks_labels = ["0", "0.5h", "1h", "1.5h", "2h / 0%"] + [f"{g:.0%}" for g in gaps_to_plot]

plt.xticks(
    ticks=x_ticks,
    labels=x_ticks_labels,
    fontsize=fontsize
)

# plt.yscale("log")
# plt.yticks(
#     ticks=[1, 10, 100, 1000], labels=["1", "10", "100", "1,000"],
#     fontsize=fontsize
# )
plt.yticks(fontsize=fontsize)

# Vertical line at timelimit
plt.axvline(x=timelimit, color="k", linestyle="--", linewidth=0.5)

plt.legend(fontsize=fontsize-1)

plt.ylabel("Number of Instances", fontsize=fontsize)
plt.xlabel("        Time                                   Optimality Gap", fontsize=fontsize)

plt.grid(True, color='lightgrey', linestyle='--', linewidth=0.5)


plt.tight_layout()
plt.savefig("figures/cumulative_plot.pdf", format="pdf", bbox_inches="tight")
plt.show()


In [ ]:
# Statistics for the paper
df_dd_solved = df[ (df["solver"] == "dd") & (df["width"] == base_width) & (df["solved"]) ]
df_bilevel_solved = df[ (df["solver"] == "bilevelsolver") & (df["solved"]) ]
df_mibs_solved = df[ (df["solver"] == "mibs") & (df["solved"]) ]

# Mean and 75% quantile of times for solved instances
for name, d in [("DD", df_dd_solved), ("B&C", df_bilevel_solved), ("MibS", df_mibs_solved)]:
    print(f"{name}: mean time = {d['time'].mean():.2f}, 75% quantile = {d['time'].quantile(0.75):.2f}")

# DD instances below 50\% optimality gap (percentage)
dd_below_50pct = df[ (df["solver"] == "dd") & (df["width"] == base_width) & (df["optgap"] <= 0.5) ]
pct_below_50pct = 100 * dd_below_50pct.shape[0] / df[ (df["solver"] == "dd") & (df["width"] == base_width) ].shape[0]
print(f"DD instances with optimality gap <= 50%: {pct_below_50pct:.2f}%")

# BC instances below 50\% optimality gap (percentage)
bc_below_50pct = df[ (df["solver"] == "bilevelsolver") & (df["optgap"] >= 0.5) ]
pct_below_50pct = 100 * bc_below_50pct.shape[0] / df[ (df["solver"] == "bilevelsolver") ].shape[0]
print(f"B&C instances with optimality gap <= 50%: {pct_below_50pct:.2f}%")

# MibS instances below 50\% optimality gap (percentage)
mibs_below_50pct = df[ (df["solver"] == "mibs") & (df["optgap"] >= 0.5) ]
pct_below_50pct = 100 * mibs_below_50pct.shape[0] / df[ (df["solver"] == "mibs") ].shape[0]
print(f"MibS instances with optimality gap <= 50%: {pct_below_50pct:.2f}%")


<b>Optimality Gap Analysis</b>

In [ ]:
# Create scatter plot comparing DD and Bilevel times

# Consider feasible instances only of DD and Bilevel
df_feasible = df_aggregated[ (df_aggregated["dd_feasible"]) & (df_aggregated["bilevelsolver_feasible"]) & (~df_aggregated["dd_solver_solved"]) & (~df_aggregated["bilevelsolver_solver_solved"]) ]

# Remove outlier
# df_feasible = df_feasible[ df_feasible["dd_optgap"] <= 1 ]
# df_feasible = df_feasible[ df_feasible["bilevelsolver_optgap"] <= 1 ]

# Add percentages
df_feasible["dd_optgap_p"] = df_feasible["dd_optgap"] * 100
df_feasible["bilevelsolver_optgap_p"] = df_feasible["bilevelsolver_optgap"] * 100

def truncate_colormap(cmap_name, minval=0.2, maxval=0.9, n=256):
    cmap = plt.get_cmap(cmap_name)
    new_cmap = mpl.colors.LinearSegmentedColormap.from_list(
        f"trunc({cmap_name},{minval:.2f},{maxval:.2f})",
        cmap(np.linspace(minval, maxval, n))
    )
    return new_cmap

# Example: darker range of PuBuGn
trunc_cmap = truncate_colormap("flare", 0.1, 0.7)

fig, ax = plt.subplots()

palette = trunc_cmap

# Scatter plot
sns.scatterplot(
    data=df_feasible, 
    x="dd_optgap_p", 
    y="bilevelsolver_optgap_p", 
    s=40,
    hue="upper_binary_variables",   # map color to this column
    palette=palette,
    ax=ax,
    legend=False
)

# Limit to 0-100% gap for visualization (actual percentages are reported in Table 2)
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)

# Add x=y line (dashed, light gray)
lims = [
    min(ax.get_xlim()[0], ax.get_ylim()[0]),
    max(ax.get_xlim()[1], ax.get_ylim()[1])
]
ax.plot(lims, lims, linestyle="--", color="lightgray")

ax.set_xlim(lims)
ax.set_ylim(lims)

ax.set_xlabel(f"{labels["dd"]} Optimality Gap (%)")
ax.set_ylabel(f"{labels["bilevelsolver"]} Optimality Gap (%)")

# Add colorbar
norm = plt.Normalize(df_feasible["state_variables"].min(),
                     df_feasible["state_variables"].max())
sm = plt.cm.ScalarMappable(cmap=palette, norm=norm)
sm.set_array([])
fig.colorbar(sm, ax=ax, label="Number of Interaction Constraints (m)")

plt.tight_layout()
plt.savefig("figures/gap_plot.pdf", format="pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Count instances to report in the paper

df_feasible = df_aggregated[ (df_aggregated["dd_feasible"]) & (df_aggregated["bilevelsolver_feasible"]) & (~df_aggregated["dd_solver_solved"]) & (~df_aggregated["bilevelsolver_solver_solved"]) ]
df_feasible["dd_optgap_p"] = df_feasible["dd_optgap"] * 100
df_feasible["bilevelsolver_optgap_p"] = df_feasible["bilevelsolver_optgap"] * 100

df_feasible[ (df_feasible["dd_optgap_p"] <= 100) & (df_feasible["bilevelsolver_optgap_p"] <= 100) ].shape[0]


In [ ]:
# Statistics for the paper

print("Number of feasible solutions found (points in the plot):", df_feasible.shape[0])

# Instance where DD has a better gap than B&C
dd_better = df_feasible[ df_feasible["dd_optgap"] < df_feasible["bilevelsolver_optgap"] ]
print(f"Instances where {labels['dd']} has a better gap than {labels['bilevelsolver']}: {dd_better.shape[0]}")

# Mean gaps and standard deviations of gaps, grouped by number of state variables
for n in df_feasible["state_variables"].unique():
    print (f"\nState Variables: {n}")
    for name, d in [("DD", df_feasible[df_feasible["state_variables"] == n]["dd_optgap"]), ("B&C", df_feasible[df_feasible["state_variables"] == n]["bilevelsolver_optgap"])]:
        print(f"\t\t{name}: mean gap = {d.mean():.4f}, std gap = {d.std():.4f}")

# # Pearson correlation coefficient between gaps and optimality gaps
# from scipy.stats import pearsonr
# for name, d in [("DD", df_feasible["dd_optgap"]), ("B&C", df_feasible["bilevelsolver_optgap"])]:
#     corr, _ = pearsonr(d, df_feasible["state_variables"])
#     print(f"Pearson correlation between {name} gap and number of state variables: {corr:.4f}")

In [ ]:
# pearsonr(d, df_feasible["state_variables"])
d

<b>Results based on classes</b>

In [ ]:
# Obtain classes
class_list = df_info_benchmark["class"].unique()


solvers = ["mibs", "bilevelsolver", "dd"]

data = {
    "classes": [],
    "type":[],
    "count": [],
}
for solver in solvers:
    data[f"{solver}_feasible"] = []
    data[f"{solver}_solved"] = []
    data[f"{solver}_time_avg"] = []
    data[f"{solver}_time_std"] = []
    data[f"{solver}_<1"] = []
    data[f"{solver}_<10"] = []
    data[f"{solver}_<100"] = []
    

for classname, dfClass in df_info_benchmark.groupby(["class", "type"]):
    data["classes"].append(classname[0])
    data["type"].append(classname[1])
    data["count"].append(dfClass.shape[0])
    
    instances = dfClass["name"].unique()
    
    for solver in solvers:
        df_solver = df[ (df["solver"] == solver) & (df["name"].isin(instances)) & (df["feasible"]) ]
        if solver == "dd":
            df_solver = df_solver[ df_solver["width"] == base_width ]
        data[f"{solver}_feasible"].append(
            df_solver.shape[0]
        )
        data[f"{solver}_solved"].append(
            df_solver[ (df_solver["solved"]) ].shape[0]
        )
        data[f"{solver}_time_avg"].append(
            df_solver[ (df_solver["solved"]) ]["time"].mean()
        )
        data[f"{solver}_time_std"].append(
            df_solver[ (df_solver["solved"]) ]["time"].std() 
        )
        data[f"{solver}_<1"].append(
            df_solver[ (~df_solver["solved"]) & (df_solver["optgap"] < 0.01) ].shape[0]
        )
        data[f"{solver}_<10"].append(
            df_solver[ (~df_solver["solved"]) & (df_solver["optgap"] < 0.1) ].shape[0]
        )
        data[f"{solver}_<100"].append(
            df_solver[ (~df_solver["solved"]) & (df_solver["optgap"] < 1) ].shape[0]
        )

df_summary_classes = pd.DataFrame(data).sort_values(by="count", ascending=False)
df_summary_classes

### Table 3: Results per classes

In [ ]:
classes = ["OR", "interKP", "weish", "K50", "general", "others/general", "others/interdiction"]

for cls in classes:
    count = df_info_benchmark[ (df_info_benchmark["class"] == cls) ].shape[0]

    result_str = rf"\texttt{{{cls}}} & {count}"
    for solver in ["mibs", "bilevelsolver", "dd"]:
        # Obtain stats
        d = df[ (df["solver"] == solver) & (df["class"] == cls) ]
        if solver == "dd":
            d = d[ d["width"] == base_width ]
        count_feasible = d[ (d["feasible"]) ].shape[0]
        count_solved   = d[ (d["solved"]) ].shape[0]
        if count_feasible == 0:
            count_feasible = "-"
            count_solved = "-"
            avg_time = "-"
            count_below_10 = "-"
            percent_below_10 = "-"
        else:
            if count_solved == 0:
                count_solved = "-"
                avg_time = "-"
                count_below_10 = "-"
                percent_below_10 = "-"
            else:
                avg_time = d[ (d["solved"]) ]["time"].mean()
                avg_time = f"{avg_time:,.1f}" 
            count_below_10  = d[ (~d["solved"]) & (d["optgap"] < 0.1) & (d["feasible"]) ].shape[0]
            percent_below_10 = 100 * count_below_10 / count_feasible
            if percent_below_10 <= 0:
                percent_below_10 = "-"
            else:
                percent_below_10 = f"{percent_below_10:.1f}" + "\\%"

        # Compose string, replacing Nan with --
        result_str += f" & {count_feasible} & {count_solved} & {avg_time} & {percent_below_10}"

    print(result_str + " \\\\")
 
    # Duplicate per variable size
    if cls != "others/general":
        mset = list(df_info_benchmark[ df_info_benchmark["class"] == cls ]["state_variables"].unique())
        mset = sorted(mset)
        for m in mset:
            count = df_info_benchmark[ (df_info_benchmark["class"] == cls) & (df_info_benchmark["state_variables"] == m) ].shape[0]
            result_str = rf"\;\;\;\;\;\;{m} & {count}"
            for solver in ["mibs", "bilevelsolver", "dd"]:
                # Obtain stats
                d = df[ (df["solver"] == solver) & (df["class"] == cls) & (df["state_variables"] == m) ]
                if solver == "dd":
                    d = d[ d["width"] == base_width ]
                count_feasible = d[ (d["feasible"]) ].shape[0]
                count_solved   = d[ (d["solved"]) ].shape[0]
                if count_feasible == 0:
                    count_feasible = "-"
                    count_solved = "-"
                    avg_time = "-"
                    count_below_10 = "-"
                    percent_below_10 = "-"
                else:
                    if count_solved == 0:
                        count_solved = "-"
                        avg_time = "-"
                        count_below_10 = "-"
                        percent_below_10 = "-"
                    else:
                        avg_time = d[ (d["solved"]) ]["time"].mean()
                        avg_time = f"{avg_time:,.1f}" 
                    count_below_10  = d[ (~d["solved"]) & (d["optgap"] < 0.1) & (d["feasible"]) ].shape[0]
                    percent_below_10 = 100 * count_below_10 / count_feasible
                    if percent_below_10 <= 0:
                        percent_below_10 = "-"
                    else:
                        percent_below_10 = f"{percent_below_10:.1f}" + "\\%"
                # Compose string, replacing Nan with --
                result_str += f" & {count_feasible} & {count_solved} & {avg_time} & {percent_below_10}"

            print(result_str + " \\\\")



In [ ]:
df_info

In [ ]:
df_summary_classes.columns

### Network Analysis: Gaps


<b>Calculate root gaps</b>

In [ ]:
# Collect all instances solved by DD 
# Note: only one extra instance was solved by bilevelsolver

# Obtain instances solved by dd
df_dd = df[ (df["solver"] == "dd") & (df["width"] == base_width) & (df["feasible"]) ].copy()

# Calculate how far mdd root gaps are from best objective calculated by the DD

df_dd["mddrootgap"] = df_dd.apply(
    lambda row: (row["bestobj"] - row["mddrootbound"]) / abs(row["bestobj"]),
    axis=1
)

df_dd["rootgap"] = df_dd.apply(
    lambda row: (row["bestobj"] - row["rootbound"]) / abs(row["bestobj"]),
    axis=1
)

print( df_dd["mddrootgap"].describe() )
print( df_dd["rootgap"].describe() )


In [ ]:
# Instances where the root gap is smaller than the final optimality gap
# This happens because the initial iteration of the solution approach is not solved to optimality

df_dd[ df_dd["optgap"] > df_dd["rootgap"] ][["name", "bestobj", "mddrootbound", "rootbound", "optgap", "rootgap", "mddrootgap", "num_iters"] ]

<b>Plots and stats</b>

In [ ]:
# Root gap plot

# Define bins
bins = [-np.inf, 100, 250, 500, np.inf]
label_bins = list(range(len(bins)-1))

# Assign bins
df_dd["state_bin"] = pd.cut(df_dd["state_variables"], bins=bins, labels=label_bins, right=False).astype(int)

# Status label
df_dd["status"] = df_dd["solved"].apply(
    lambda s: "Closed" if s else "Open"
)

# Percentage
df_dd["pct_mddrootgap"] = df_dd["mddrootgap"]*100

# Ensure consistent order of the two hue levels
hue_order = ["Closed", "Open"]  # or list(df_dd["status"].unique()) in your desired order

# Take the next two colors from the default 'deep' palette: index 2 (green), 3 (red)
deep = sns.color_palette("deep")
palette = {hue_order[0]: deep[3],  # green
           hue_order[1]: deep[4]}  # red

ax = sns.boxplot(
    data=df_dd,
    x="state_bin",
    y="pct_mddrootgap",
    hue="status",
    hue_order=hue_order,
    palette=palette
)

# Your existing ticks/labels
plt.xticks([0, 1, 2, 3], ["<100", "100-249", "250-499", "500"], fontsize=fontsize)
plt.yticks(fontsize=fontsize)
plt.xlabel("Number of Interaction Constraints (m)", fontsize=fontsize)
plt.ylabel("DD Root Gap (%)", fontsize=fontsize)

# Legend just to the right — adjust 1.02 to 1.04/1.06 if you want a bit more space
# plt.legend(loc="upper left", fontsize=fontsize, title=None)
ax.legend(loc="upper left", bbox_to_anchor=(0.0, 0.94), fontsize=fontsize, title=None)

# Make room on the right for the legend
plt.tight_layout()
plt.savefig("figures/mddrootgap.pdf", format="pdf", bbox_inches="tight")

plt.show()

In [ ]:
# Print average root gaps by status and state variable bin
for (status, state_bin), group in df_dd.groupby(["status", "state_bin"]):
    avg_gap = group["mddrootgap"].mean() * 100
    count = group.shape[0]
    print(f"Status: {status}, State Bin: {state_bin}, Count: {count}, Average MDD Root Gap: {avg_gap:.2f}%")

In [ ]:

# Aggregate all instances with state_variables < 100 into one bucket
df_dd["state_var_group"] = df_dd["state_variables"].apply(
    lambda x: "<100" if x < 100 else str(int(x))
)

# Create bins for state variables
bins = [-np.inf, 100, 250, 500, np.inf]
df_dd["state_bin"] = pd.cut(df_dd["state_variables"], bins=bins, labels=label_bins, right=False).astype(int)

# Sort x in the desired order: "<100", then numeric ascending
def sort_key(v):
    return -1 if v == "<100" else int(v)
df_dd["sort_key"] = df_dd["state_var_group"].apply(sort_key)
df_dd = df_dd.sort_values("sort_key")

# Restrict to knapsack problems
df_class = df_dd[ df_dd["class2"] == "knapsack" ]
# df_class = df_dd

# % Columns
# df_class["pct_hprgap"] = df_class["hprgap"]*100
df_class["pct_mddrootgap"] = df_class["mddrootgap"]*100
df_class["pct_rootgap"] = df_class["rootgap"]*100
df_class["pct_optgap"] = df_class["optgap"]*100


sns.lineplot(
    data=df_class, 
    x="state_var_group", 
    y="pct_mddrootgap", 
    label="DD Root Gap",
    marker='o'
)

# sns.lineplot(
#     data=df_class, 
#     x="state_var_group", 
#     y="pct_rootgap", 
#     label="DD+Cuts Root Gap",
#     marker='X'
# )

sns.lineplot(
    data=df_class, 
    x="state_var_group", 
    y="pct_optgap", 
    label="Final Opt Gap",
    marker='D',
    markersize=4
)

# sns.lineplot(
#     data=df_class, 
#     x="state_var_group", 
#     y="pct_hprgap", 
#     label="HPR+Cuts Root Gap",
#     marker='X',
#     markersize=4
# )

plt.xlabel("Number of Interaction Constraints (m)", fontsize=fontsize)
plt.ylabel("Gap (%) w.r.t. Best-known DD Solution", fontsize=fontsize)

plt.xticks(fontsize=fontsize)
plt.yticks(fontsize=fontsize)

plt.tight_layout()
plt.savefig("figures/knapsack_rootgap.pdf", format="pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Stats for paper
df_class.groupby("state_bin").agg(
    count_instances = ("name", "count"),
    avg_mddrootgap = ("mddrootgap", "mean"),
    std_mddrootgap = ("mddrootgap", "std"),
    avg_optgap = ("optgap", "mean"),
    std_optgap = ("optgap", "std"),
).reset_index()

In [ ]:
print( df_class["optgap"].describe() )
print( df_class["mddrootgap"].describe() )

In [ ]:
# Aggregate all instances with state_variables < 100 into one bucket
df_dd["state_var_group"] = df_dd["state_variables"].apply(
    lambda x: "<100" if x < 100 else str(int(x))
)

# Create bins for state variables
bins = [-np.inf, 100, 250, 500, np.inf]
df_dd["state_bin"] = pd.cut(df_dd["state_variables"], bins=bins, labels=label_bins, right=False).astype(int)

# Sort x in the desired order: "<100", then numeric ascending
def sort_key(v):
    return -1 if v == "<100" else int(v)
df_dd["sort_key"] = df_dd["state_var_group"].apply(sort_key)
df_dd = df_dd.sort_values("sort_key")

# Restrict to knapsack problems
df_class = df_dd[ df_dd["class2"] == "multidimensional knapsack" ]
# df_class = df_dd

# % Columns
# df_class["pct_hprgap"] = df_class["hprgap"]*100
df_class["pct_mddrootgap"] = df_class["mddrootgap"]*100
df_class["pct_rootgap"] = df_class["rootgap"]*100
df_class["pct_optgap"] = df_class["optgap"]*100


sns.lineplot(
    data=df_class, 
    x="state_var_group", 
    y="pct_mddrootgap", 
    label="DD Root Gap",
    marker='o'
)

# sns.lineplot(
#     data=df_class, 
#     x="state_var_group", 
#     y="pct_rootgap", 
#     label="DD+Cuts Root Gap",
#     marker='X'
# )

sns.lineplot(
    data=df_class, 
    x="state_var_group", 
    y="pct_optgap", 
    label="Final Opt Gap",
    marker='D',
    markersize=4
)

# sns.lineplot(
#     data=df_class, 
#     x="state_var_group", 
#     y="pct_hprgap", 
#     label="HPR+Cuts Root Gap",
#     marker='X',
#     markersize=4
# )

plt.xlabel("Number of Interaction Constraints (m)", fontsize=fontsize)
plt.ylabel("Gap (%) w.r.t. Best-known DD Solution", fontsize=fontsize)

plt.xticks(fontsize=fontsize)
plt.yticks(fontsize=fontsize)

plt.tight_layout()
plt.savefig("figures/multiknapsack_rootgap.pdf", format="pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Stats for paper
print( df_class["optgap"].describe() )
print( df_class["mddrootgap"].describe() )

In [ ]:
# # Aggregate all instances with state_variables < 100 into one bucket
# df_dd["state_var_group"] = df_dd["state_variables"].apply(
#     lambda x: "<100" if x < 100 else str(int(x))
#     # lambda x: str(int(x))
# )

# Aggregate all instances with state_variables < 100 into one bucket
df_dd["state_var_group"] = df_dd["state_variables"].apply(
    lambda x: "<50" if x < 50 else "50-100" if x < 100 else ">100"
    # lambda x: str(int(x))
)

# Create bins for state variables
bins = [-np.inf, 100, 250, 500, np.inf]
df_dd["state_bin"] = pd.cut(df_dd["state_variables"], bins=bins, labels=label_bins, right=False).astype(int)

# # Sort x in the desired order: "<100", then numeric ascending
# def sort_key(v):
#     return -1 if v == "<100" else int(v)
# df_dd["sort_key"] = df_dd["state_var_group"].apply(sort_key)
# df_dd = df_dd.sort_values("sort_key")

# Sort x in the desired order: "<100", then numeric ascending
def sort_key(v):
    return -1 if v == "<50" else 0 if v == "50-100" else 1
df_dd["sort_key"] = df_dd["state_var_group"].apply(sort_key)
df_dd = df_dd.sort_values("sort_key")


# Restrict to knapsack problems
df_class = df_dd[ (df_dd["class2"] != "multidimensional knapsack") & (df_dd["class2"] != "knapsack") ]
# df_class = df_dd

# % Columns
# df_class["pct_hprgap"] = df_class["hprgap"]*100
df_class["pct_mddrootgap"] = df_class["mddrootgap"]*100
df_class["pct_rootgap"] = df_class["rootgap"]*100
df_class["pct_optgap"] = df_class["optgap"]*100


sns.lineplot(
    data=df_class, 
    x="state_var_group", 
    y="pct_mddrootgap", 
    label="DD Root Gap",
    marker='o'
)

# sns.lineplot(
#     data=df_class, 
#     x="state_var_group", 
#     y="pct_rootgap", 
#     label="DD+Cuts Root Gap",
#     marker='X'
# )

sns.lineplot(
    data=df_class, 
    x="state_var_group", 
    y="pct_optgap", 
    label="Final Opt Gap",
    marker='D',
    markersize=4
)

# sns.lineplot(
#     data=df_class, 
#     x="state_var_group", 
#     y="pct_hprgap", 
#     label="HPR+Cuts Root Gap",
#     marker='X',
#     markersize=4
# )

plt.xlabel("Number of Interaction Constraints (m)", fontsize=fontsize)
plt.ylabel("Gap (%) w.r.t. Best-known DD Solution", fontsize=fontsize)

plt.xticks(fontsize=fontsize)
plt.yticks(fontsize=fontsize)

# plt.ylim(0, 40)

plt.tight_layout()
plt.savefig("figures/others_rootgap.pdf", format="pdf", bbox_inches="tight")
plt.show()

In [ ]:
df_class["pct_optgap"].describe()

### Network Analysis: Times and Reduction

In [ ]:
# First, because the runs have not finished, root times can be NULL; please check in the final version
df_dd[ df_dd["roottime"].isnull() ][["name", "width", "dd_time_before_first_mip", "mddroottime", "roottime"]]

In [ ]:
# SNS Color palette
deep = sns.color_palette("deep")

# Root time

# DD not solved but feasible
df_dd = df[ (df["solver"] == "dd") & (df["feasible"]) & (df["width"] == base_width) ].copy()

# Consider only knapsack / multidimensional knapsack
df_dd = df_dd[ df_dd["class2"].isin(["knapsack", "multidimensional knapsack"]) ] 

# Define bins
bins = [-np.inf, 100, 250, 500, np.inf]
label_bins = list(range(len(bins)-1))

# Assign bins
df_dd["state_bin"] = pd.cut(df_dd["state_variables"], bins=bins, labels=label_bins, right=False).astype(int)

# Percentage
df_dd["pct_optgap"] = df_dd["optgap"]*100

# Type label
df_dd["typelabel"] = df_dd["class2"].apply(
    lambda s: "Mult. knapsack" if s == "multidimensional knapsack" else "Knapsack" if s == "knapsack" else "Other"
)

# Collors
palette = {"Knapsack": deep[7],  # green
           "Mult. knapsack": deep[8],  # red
           "Other": deep[2]  # blue
}


ax = sns.lineplot(
    data=df_dd, 
    x="state_bin", 
    y="mddroottime", 
    hue="typelabel",
    style="typelabel",
    markers={"Knapsack":"D", "Mult. knapsack":"X"},
    palette=palette,
    hue_order=["Knapsack", "Mult. knapsack"]
)

# ax = sns.lineplot(
#     data=df_dd, 
#     x="state_variables", 
#     # y="pct_optgap", 
#     y="mddrootgap", 
#     hue="class",
#     # hue_order=["Knapsack", "Mult. knapsack"]
# )

plt.xticks(
    [0, 1, 2, 3],
    ["<100", "100-249", "250-499", "500"],
    fontsize=fontsize
)
plt.yticks(
    fontsize=fontsize
)

plt.xlabel("Number of Interaction Constraints (m)")
plt.ylabel("DD Root Time (s)")

# Legend just to the right — adjust 1.02 to 1.04/1.06 if you want a bit more space
plt.legend(loc="upper left", fontsize=fontsize, title=None)
# ax.legend(loc="upper left", bbox_to_anchor=(0.0, 0.94), fontsize=fontsize, title=None)

plt.tight_layout()
plt.savefig("figures/dd_root_time_class.pdf", format="pdf", bbox_inches="tight")

plt.show()


In [ ]:

# DD not solved but feasible
df_dd = df[ (df["solver"] == "dd") & (df["feasible"]) & (df["width"] == base_width) ].copy()

# Consider only knapsack / multidimensional knapsack
df_dd = df_dd[ df_dd["class2"].isin(["knapsack", "multidimensional knapsack"]) ] 

# Define bins
bins = [-np.inf, 100, 250, 500, np.inf]
label_bins = list(range(len(bins)-1))

# Assign bins
df_dd["state_bin"] = pd.cut(df_dd["state_variables"], bins=bins, labels=label_bins, right=False).astype(int)


# 1) Aggregate (mean) by state_bin. Use your preferred estimator.
g = (df_dd
     .groupby("state_bin", as_index=False)
     .agg(dd_build=("dd_time_before_first_mip", "mean"),
          root_total=("roottime", "mean")))

# 2) Compute the remainder (time after DD build but still within root)
g["root_remainder"] = (g["root_total"] - g["dd_build"]).clip(lower=0)

g = g.sort_values("state_bin")

# 4) Stacked area plot over categorical x
x = np.arange(len(g))
labels = g["state_bin"].astype(str).tolist()

fig, ax = plt.subplots()

# First layer: DD build time
ax.fill_between(x, 0, g["dd_build"], alpha=0.5, step=None)

# Second layer: remainder up to total root time
ax.fill_between(x, g["dd_build"], g["dd_build"] + g["root_remainder"],
                alpha=0.5, step=None)

# Optional outline of total
ax.plot(x, g["root_total"], lw=1.5, linestyle="-", label="DD Root Time", color=deep[1], marker='o')
ax.plot(x, g["dd_build"], lw=1.5, linestyle="-", label="Network Compilation Time", color=deep[0], marker='x')

plt.xticks(
    [0, 1, 2, 3],
    ["<100", "100-249", "250-499", "500"],
    fontsize=fontsize
)
plt.yticks(
    fontsize=fontsize
)

plt.xlabel("Number of Interaction Constraints (m)")
plt.ylabel("Time (s)")

# # Legend just to the right — adjust 1.02 to 1.04/1.06 if you want a bit more space
plt.legend(loc="upper left", fontsize=fontsize, title=None)
# # ax.legend(loc="upper left", bbox_to_anchor=(0.0, 0.94), fontsize=fontsize, title=None)

plt.tight_layout()
plt.savefig("figures/dd_root_time.pdf", format="pdf", bbox_inches="tight")

plt.show()

In [ ]:
# For paper: calculate time spent in DD build as percentage of total root time
df_dd["pct_construction"] = 100 * df_dd["dd_time_before_first_mip"] / df_dd["roottime"]
df_dd["pct_construction"].describe()

### Budget Analysis

In [ ]:
# Identify cases with monotonous decrease in root gap and iterations as budget increases, for internal analysis and future work discussion

df_dd = df[ (df["solver"] == "dd") & (df["solved"])]
# df_dd = df[ (df["solver"] == "dd") ]

for instance_name, df_dd_instance in df_dd.groupby("name"):
    # Collect state variables
    state_vars = df_dd_instance["state_variables"].unique()[0]

    # Sort by width
    df_dd_instance_sorted = df_dd_instance.sort_values(by="width")

    # Calculate mddrootgap
    df_dd_instance_sorted["mddrootgap"] = (df_dd_instance_sorted["bestobj"] - df_dd_instance_sorted["mddrootbound"]) / (np.abs(df_dd_instance_sorted["bestobj"]) + 1e-10)

    # Collect data
    gaps = list(df_dd_instance_sorted["mddrootgap"])
    bounds = list(df_dd_instance_sorted["mddrootbound"])
    mddtimes = list(df_dd_instance_sorted["mddroottime"])
    times = list(df_dd_instance_sorted["time"])
    iters = list(df_dd_instance_sorted["num_iters"])
    widths = list(df_dd_instance_sorted["width"])
    bestobjs = list(df_dd_instance_sorted["bestobj"])

    # valid = (len(gaps) >= 2)
    valid = (len(gaps) == 4) and (iters[0] > 10)

    # # Check if gaps are decreasing
    # for i in range(1, len(gaps)):
    #     if gaps[i] > gaps[i-1]:
    #         valid = False
    #         break

    # Check if iterations are decreasing
    for i in range(1, len(iters)):
        if iters[i] > iters[i-1]:
            valid = False
            break

    # # Check if bounds are decreasing
    # for i in range(1, len(bounds)):
    #     if bounds[i] <= bounds[i-1]:
    #         valid = False
    #         break

    if valid:
    # if True:
        print(f"Instance {instance_name} - m={state_vars}:")
        print(f"\tBounds: {bounds}")
        print(f"\tGaps: {gaps}")
        print(f"\tTimes: {times}")
        print(f"\tMDD times: {mddtimes}")
        print(f"\tIters: {iters}")
        print(f"\tWidths: {widths}")
        print(f"\tBestobjs: {bestobjs}")

In [ ]:
# Pick one of the instances above
instance_name = "weish22.dat-100"
df_dd = df[ (df["solver"] == "dd") & (df["solved"])]
df_dd_instance = df_dd[df_dd["name"] == instance_name]

# Sort by width
df_dd_instance_sorted = df_dd_instance.sort_values(by="width")

# Calculate mddrootgap
df_dd_instance_sorted["mddrootgap"] = (df_dd_instance_sorted["bestobj"] - df_dd_instance_sorted["mddrootbound"]) / (np.abs(df_dd_instance_sorted["bestobj"]) + 1e-10)

# Collect data
gaps = list(df_dd_instance_sorted["mddrootgap"])
bounds = list(df_dd_instance_sorted["mddrootbound"])
mddtimes = list(df_dd_instance_sorted["mddroottime"])
times = list(df_dd_instance_sorted["time"])
iters = list(df_dd_instance_sorted["num_iters"])
widths = list(df_dd_instance_sorted["width"])
bestobjs = list(df_dd_instance_sorted["bestobj"])

print(f"Instance {instance_name} - m={state_vars}:")
print(f"\tBounds: {bounds}")
print(f"\tGaps: {gaps}")
print(f"\tTimes: {times}")
print(f"\tMDD times: {mddtimes}")
print(f"\tIters: {iters}")
print(f"\tWidths: {widths}")
print(f"\tBestobjs: {bestobjs}")

In [ ]:
# Time 

fig, ax = plt.subplots()

# First layer: DD build time
ax.fill_between(widths, 0, mddtimes, alpha=0.5, step=None, color=deep[1])

# Second layer: remainder up to total root time
ax.fill_between(widths, mddtimes, times,
                alpha=0.5, step=None, color=deep[0])

# # Optional outline of total
ax.plot(widths, times, lw=1.5, linestyle="-", label="Solution Time", color=deep[0], marker='o')
ax.plot(widths, mddtimes, lw=1.5, linestyle="-", label="DD Root Time", color=deep[1], marker='x')

plt.xticks(
    [8, 16, 32, 64],
    ["8", "16", "32", "64"],
    fontsize=fontsize
)
plt.yticks(
    fontsize=fontsize
)

plt.xlabel("Budget (B)")
plt.ylabel("Time (s)")

# # # Legend just to the right — adjust 1.02 to 1.04/1.06 if you want a bit more space
plt.legend(loc="upper left", fontsize=fontsize, title=None)
# # # ax.legend(loc="upper left", bbox_to_anchor=(0.0, 0.94), fontsize=fontsize, title=None)

plt.tight_layout()
plt.savefig("figures/dd_weish_width.pdf", format="pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Number of iterations

plt.plot(widths, iters, lw=1.5, linestyle="-", label="DD Root Time", color=deep[3], marker='x')

plt.xticks(
    [8, 16, 32, 64],
    ["8", "16", "32", "64"],
    fontsize=fontsize
)
plt.yticks(
    fontsize=fontsize
)

plt.xlabel("Budget (B)")
plt.ylabel("Number of Iterations until Optimality")

# # # Legend just to the right — adjust 1.02 to 1.04/1.06 if you want a bit more space
# plt.legend(loc="upper left", fontsize=fontsize, title=None)
# # # ax.legend(loc="upper left", bbox_to_anchor=(0.0, 0.94), fontsize=fontsize, title=None)

plt.tight_layout()
plt.savefig("figures/dd_weish_iterations.pdf", format="pdf", bbox_inches="tight")
plt.show()

### Reduction/Symmetry Elimination Analysis

In [ ]:


# Obtain DD
df_dd = df[ (df["solver"] == "dd") & (df["feasible"]) & (df["width"] == base_width) ].copy()

# Calculate reduction rate 
df_dd["reduction_rate"] = df_dd["dd_num_nodes_before"]/df_dd["dd_num_nodes_after"]

# Assign bins
df_dd["state_bin"] = pd.cut(df_dd["state_variables"], bins=bins, labels=label_bins, right=False).astype(int)

# Type label
df_dd["typelabel"] = df_dd["class2"].apply(
    lambda s: "Mult. knapsack" if s == "multidimensional knapsack" else "Knapsack" if s == "knapsack" else "Other"
)

# Consider only knapsack / multidimensional knapsack
df_dd = df_dd[ df_dd["class2"].isin(["knapsack", "multidimensional knapsack"]) ] 

sns.lineplot(
    data=df_dd,
    x="state_bin",
    y="dd_num_nodes_before",
    label="Before Reduction",
    marker='X',
    markersize=7
    # hue="typelabel",
    # style="typelabel",
    # hue_order=["Knapsack", "Mult. knapsack"],
    # markers={"Knapsack":"D", "Mult. knapsack":"X"},
)

sns.lineplot(
    data=df_dd,
    x="state_bin",
    y="dd_num_nodes_after",
    label="After Reduction",
    marker='o',
    markersize=7
    # hue="typelabel",
    # style="typelabel",
    # hue_order=["Knapsack", "Mult. knapsack"],
    # markers={"Knapsack":"D", "Mult. knapsack":"X"},
)

# # Long format
# df_long = df_dd.melt(
#     id_vars="state_bin",
#     value_vars=["dd_num_nodes_before", "dd_num_nodes_after"],
#     var_name="phase",
#     value_name="dd_totalnodes",
# )
# ax = sns.boxplot(
#     data=df_long,
#     x="state_bin",
#     y="dd_totalnodes",
#     hue="phase"
# )

# ax = sns.boxplot(
#     data=df_dd,
#     x="state_bin",
#     y="reduction_rate",
#     hue="typelabel",
#     hue_order=["Knapsack", "Mult. knapsack"],
# )

plt.xticks(
    [0, 1, 2, 3],
    ["<100", "100-249", "250-499", "500"],
    fontsize=fontsize
)
plt.yticks(
    fontsize=fontsize
)

plt.xlabel("Number of Interaction Constraints (m)")
plt.ylabel("Average Number of DD nodes")

plt.legend(loc="upper left", fontsize=fontsize, title=None)

plt.tight_layout()
plt.savefig("figures/dd_reduction_nodes.pdf", format="pdf", bbox_inches="tight")

plt.show()


<b>Root node analysis</b>

In [ ]:
# Collect all instances solved by DD 
# Note: only one extra instance was solved by bilevelsolver

# Obtain instances solved by dd
df_solved = df[ (df["solver"] == "dd") & (df["solved"]) ]
# df_solved = df[ (df["solver"] == "dd") & (df["feasible"]) ]

# TODO: run missing instance
df_solved = df_solved[ ~df_solved["mddrootbound"].isnull() ]

# # Calculate how far mdd roots are from best known
# df_solved["mddrootgap"] = df_solved.apply(
#     lambda row: (row["min_bestobj"] - row["mddrootbound"]) / abs(row["min_bestobj"]),
#     axis=1
# )

# Calculate how far mdd roots are from best known
df_solved["mddrootgap"] = df_solved.apply(
    lambda row: (row["bestobj"] - row["mddrootbound"]) / abs(row["bestobj"]),
    axis=1
)

df_solved["mddrootgap"].describe()

In [ ]:
# Pick KP instances

df_kp = df_solved[ df_solved["name"].str.contains("interKP") ]

# Split the name
parts = df_kp["name"].str.split("-", expand=True)

# Assign columns (skip the prefix "interKP")
df_kp["interKP_n"] = parts[1].astype(int)
df_kp["interKP_m"] = parts[2].astype(int)
df_kp["interKP_B"] = parts[3].astype(int)
df_kp["interKP_K"] = parts[4].astype(int)

df_kp

In [ ]:
# Pick OR instances

import re
df_or = df_solved[ df_solved["name"].str.contains("OR") ]

_pat_or = re.compile(
    r"""^OR
        (?P<or_knapsacks>\d+)      # 10
        x
        (?P<or_upper_vars>\d+)     # 100
        -
        (?P<or_rhs_scale>\d+(?:\.\d+)?)  # 0.25
        (?:_(?P<or_seed>\d+))?           # optional _2
        (?:\.dat-(?P<or_suffix>\d+))?    # optional .dat-50
        $""",
    re.VERBOSE
)

def parse_or_name(s: str):
    m = _pat_or.match(s)
    if not m:
        return pd.Series(
            {
                "or_knapsacks": None,
                "or_upper_vars": None,
                "or_rhs_scale": None,
                "or_seed": None,
                "or_suffix": None,
            }
        )
    g = m.groupdict()
    return pd.Series(
        {
            "or_knapsacks": int(g["or_knapsacks"]),
            "or_upper_vars": int(g["or_upper_vars"]),
            "or_rhs_scale": float(g["or_rhs_scale"]),
            "or_seed": int(g["or_seed"]) if g["or_seed"] is not None else None,
            "or_suffix": int(g["or_suffix"]) if g["or_suffix"] is not None else None,
        }
    )

# Extract rows
df_or[["or_knapsacks","or_upper_vars","or_rhs_scale","or_seed","or_suffix"]]= df_or["name"].apply(parse_or_name)

df_or.head(10)

MDD Root Gap Analysis

In [ ]:
# Calculate root bound gap

df_dd = df[ df["solver"] == "dd" ]

# Calculate how far mdd roots are from best known
df_dd["mddrootgap"] = df_dd.apply(
    lambda row: (row["bestobj"] - row["mddrootbound"]) / abs(row["bestobj"]),
    axis=1
)

df_dd["mddrootgap"].describe()

In [ ]:
df_test = df[ (df["solver"] == "dd") & (df["solved"]) ].sort_values(by="mddrootgap", ascending=False)[["name", "state_variables", "class", "optgap", "time", "mddrootgap", "mddrootbound", "mddroottime", "bestobj", "num_iters"]]
df_test[df_test["time"] < 9000].sort_values(by="time", ascending=False).head(10)
# df_test[ df_test["class"] == "interKP"]
# df_test[ df_test["name"] == "interKP-100-100-1-1"] 


In [ ]:
df[ (df["solver"] == "dd") & (~df["solved"]) ][["name", "state_variables", "optgap", "bestbound", "time", "rootbound", "mddrootgap", "mddrootbound", "mddroottime", "bestobj"]]

In [ ]:
# solved instances
df[ ((df["solver"] == "dd") | (df["solver"] == "dd_nologicalcuts")) & df["solved"] ].groupby("name").min().reset_index()[["name", "time"]].to_csv("solved_instances.csv", index=False)

<b>Check width improvement/changes</b>

In [ ]:
df_dd = df[ (df["solver"] == "dd") & (df["solved"])]
# df_dd = df[ (df["solver"] == "dd") ]

for instance_name, df_dd_instance in df_dd.groupby("name"):
    # Sort by width
    df_dd_instance_sorted = df_dd_instance.sort_values(by="width")

    # Calculate mddrootgap
    df_dd_instance_sorted["mddrootgap"] = (df_dd_instance_sorted["bestobj"] - df_dd_instance_sorted["mddrootbound"]) / (np.abs(df_dd_instance_sorted["bestobj"]) + 1e-10)

    # Collect data
    gaps = list(df_dd_instance_sorted["mddrootgap"])
    bounds = list(df_dd_instance_sorted["mddrootbound"])
    mddtimes = list(df_dd_instance_sorted["mddroottime"])
    times = list(df_dd_instance_sorted["time"])
    iters = list(df_dd_instance_sorted["num_iters"])
    widths = list(df_dd_instance_sorted["width"])
    bestobjs = list(df_dd_instance_sorted["bestobj"])

    # valid = (len(gaps) >= 2)
    valid = (len(gaps) == 4) and (iters[0] > 10)

    # # Check if gaps are decreasing
    # for i in range(1, len(gaps)):
    #     if gaps[i] > gaps[i-1]:
    #         valid = False
    #         break

    # Check if iterations are decreasing
    for i in range(1, len(iters)):
        if iters[i] >= iters[i-1]:
            valid = False
            break

    # # Check if bounds are decreasing
    # for i in range(1, len(bounds)):
    #     if bounds[i] <= bounds[i-1]:
    #         valid = False
    #         break

    if valid:
    # if True:
        print(f"Instance {instance_name}:")
        print(f"\tBounds: {bounds}")
        print(f"\tGaps: {gaps}")
        print(f"\tTimes: {times}")
        print(f"\tMDD times: {mddtimes}")
        print(f"\tIters: {iters}")
        print(f"\tWidths: {widths}")
        print(f"\tBestobjs: {bestobjs}")


### Analysis: Augmented MibS+DD and Bilevel+DD

In [ ]:
# Date augmented results were run
run_date = "Mar24_2026"

In [ ]:
# Read results files
df_augmented = pd.read_csv(f"results_solvers_plus_dd_{run_date}_mibs_bc_augmented.csv")
df_augmented.head(3)

In [ ]:
# Pre-clean

# If it has feasible solution
df_augmented["feasible"] = ~df_augmented["bestobj"].isna()

# If instance is solved to optimality
opt_threshold = 1e-10
df_augmented["solved"] = (df_augmented["optgap"].abs() < opt_threshold) & df_augmented["feasible"]


In [ ]:
# Width to compare
based_width_augmented = 8

In [ ]:
# Remove filtered out df instances based on discrete variables
df_f = df[ (df["lower_continuous_variables"] == 0) & (df["lower_integer_variables"] == 0) & (~df["has_noninteger_coefficients"]) ]
len(df_f["name"].unique())

In [ ]:
# Create aggregated dataframe with new data

# Merge dataframes: only consider base width for DD
dfDD = df_f[ (df_f["solver"] == "dd") & (df_f["width"] == base_width) ]
dfMibs = df_f[df_f["solver"] == "mibs"]
dfBilevel = df_f[df_f["solver"] == "bilevelsolver"]
dfMibsDD = df_augmented[(df_augmented["solver"] == "mibs_plus_dd") & (df_augmented["width"] == based_width_augmented)]
dfBilevelDD = df_augmented[(df_augmented["solver"] == "bilevelsolver_plus_dd") & (df_augmented["width"] == based_width_augmented)]

dfDD_pref = (
    dfDD
    .rename(columns={
        "bestobj"                : "dd_bestobj",
        "time"                   : "dd_time",
        "optgap"                 : "dd_optgap",
        "solved"                 : "dd_solver_solved",
        "feasible"               : "dd_feasible",
        "solved"                 : "dd_solved",
    })
    .drop(columns=["solver"])
)

dfBilevel_pref = (
    dfBilevel
    .rename(columns={
        "bestobj"                : "bilevelsolver_bestobj",
        "time"                   : "bilevelsolver_time",
        "optgap"                 : "bilevelsolver_optgap",
        "solved"                 : "bilevelsolver_solver_solved",
        "feasible"               : "bilevelsolver_feasible",
        "solved"                 : "bilevelsolver_solved",
    })
    .drop(columns=["solver"])
)

dfMibs_pref = (
    dfMibs
    .rename(columns={
        "bestobj"                    : "mibs_bestobj",
        "time"                       : "mibs_time",
        "optgap"                     : "mibs_optgap",
        "solved"                     : "mibs_solver_solved",
        "feasible"                   : "mibs_feasible",
        "solved"                     : "mibs_solved",
    })
    .drop(columns=["solver"])
)

dfMibsDD_pref = (
    dfMibsDD
    .rename(columns={
        "bestobj"                    : "mibs_plus_dd_bestobj",
        "time"                       : "mibs_plus_dd_time",
        "optgap"                     : "mibs_plus_dd_optgap",
        "solved"                     : "mibs_plus_dd_solver_solved",
        "feasible"                   : "mibs_plus_dd_feasible",
        "width"                      : "mibs_plus_dd_width",
        "solved"                     : "mibs_plus_dd_solved",
    })
    .drop(columns=["solver"])
)

dfBilevelDD_pref = (
    dfBilevelDD
    .rename(columns={
        "bestobj"                    : "bilevelsolver_plus_dd_bestobj",
        "time"                       : "bilevelsolver_plus_dd_time",
        "optgap"                     : "bilevelsolver_plus_dd_optgap",
        "solved"                     : "bilevelsolver_plus_dd_solver_solved",
        "feasible"                   : "bilevelsolver_plus_dd_feasible",
        "width"                      : "bilevelsolver_plus_dd_width",
        "solved"                     : "bilevelsolver_plus_dd_solved",
    })
    .drop(columns=["solver"])
)


# Merge & validate one-to-one relationship
key_cols = ["name"]
df_aggregated = (
    dfDD_pref
    .merge(dfBilevel_pref, on=key_cols, how="outer", validate="one_to_one", suffixes=("", "_bilevel"))
    .merge(dfMibs_pref,   on=key_cols, how="outer", validate="one_to_one", suffixes=("", "_mibs"))
    .merge(dfMibsDD_pref, on=key_cols, how="outer", validate="one_to_one", suffixes=("", "_mibsdd"))
    .merge(dfBilevelDD_pref, on=key_cols, how="outer", validate="one_to_one", suffixes=("", "_bileveldd"))
)


# Empty feasible indicates no feasible solution found
df_aggregated["dd_feasible"] = df_aggregated["dd_feasible"].astype("boolean").fillna(False)
df_aggregated["bilevelsolver_feasible"] = df_aggregated["bilevelsolver_feasible"].astype("boolean").fillna(False)
df_aggregated["mibs_feasible"] = df_aggregated["mibs_feasible"].astype("boolean").fillna(False)
df_aggregated["mibs_plus_dd_feasible"] = df_aggregated["mibs_plus_dd_feasible"].astype("boolean").fillna(False)
df_aggregated["bilevelsolver_plus_dd_feasible"] = df_aggregated["bilevelsolver_plus_dd_feasible"].astype("boolean").fillna(False)

df_aggregated["dd_solved"] = df_aggregated["dd_solved"].astype("boolean").fillna(False)
df_aggregated["bilevelsolver_solved"] = df_aggregated["bilevelsolver_solved"].astype("boolean").fillna(False)
df_aggregated["mibs_solved"] = df_aggregated["mibs_solved"].astype("boolean").fillna(False)
df_aggregated["bilevelsolver_plus_dd_solved"] = df_aggregated["bilevelsolver_plus_dd_solved"].astype("boolean").fillna(False)
df_aggregated["mibs_plus_dd_solved"] = df_aggregated["mibs_plus_dd_solved"].astype("boolean").fillna(False)

# columns = [col for col in df_aggregated.columns.tolist() if "dd" in col or "bilevelsolver" in col or "mibs" in col or "name" in col or "width" in col or "class" in col or "class2" in col or "type" in col or "known_objective_value" in col or "state" in col or ""]

# df_aggregated[columns]


Summary of Solver Results

In [ ]:
# Collect solvers
dfDD = df_f[ (df_f["solver"] == "dd") & (df_f["width"] == 8) ]
dfBilevel = df_f[ (df_f["solver"] == "bilevelsolver") ]
dfMibs = df_f[ (df_f["solver"] == "mibs") ]
dfMibsDD = df_augmented[(df_augmented["solver"] == "mibs_plus_dd") & (df_augmented["width"] == based_width_augmented)]
dfBilevelDD = df_augmented[(df_augmented["solver"] == "bilevelsolver_plus_dd") & (df_augmented["width"] == based_width_augmented)]

summary_results = {
    "Method": [],
    "Proved Optimality": [],
    "Has Feasible Solution": [],
    "% Gap <= 1%": [],
    "% Gap <= 10%": [],
    "% Gap <= 100%": [],
    "Better DD": [],
    "Better Bilevel": [],
    "Better Mibs": [],
    "Better Bilevel+DD": [],
    "Better Mibs+DD": [],
}

for method, df_method in [("bilevelsolver", dfBilevel), ("bilevelsolver_plus_dd", dfBilevelDD), ("mibs", dfMibs), ("mibs_plus_dd", dfMibsDD), ("dd", dfDD)]:
    # General stats
    summary_results["Method"].append(method)
    summary_results["Proved Optimality"].append(int(df_method["solved"].sum()))
    summary_results["Has Feasible Solution"].append(int(df_method["feasible"].sum()))
    summary_results["% Gap <= 1%"].append(
        round(
            100* (df_method[ (df_method["feasible"]) & (df_method["optgap"] <= 0.01)].shape[0])/(df_method[ (df_method["feasible"])].shape[0])
        ,2)
    )
    summary_results["% Gap <= 10%"].append(
        round(
            100* (df_method[ (df_method["feasible"]) & (df_method["optgap"] <= 0.1)].shape[0])/(df_method[ (df_method["feasible"])].shape[0])
        ,2)
    )
    summary_results["% Gap <= 100%"].append(
        round(
            100* (df_method[ (df_method["feasible"]) & (df_method["optgap"] <= 1)].shape[0])/(df_method[ (df_method["feasible"])].shape[0])
        ,2)
    )
    
    # Comparison statistics
    summary_results["Better DD"].append(
        int( 
            ( (df_aggregated[f"{method}_feasible"]) & ( (~df_aggregated["dd_feasible"]) | (df_aggregated[f"{method}_bestobj"] <= df_aggregated["dd_bestobj"]-0.1) ) ).sum()  
        ) if method != "dd" else np.nan
    )

    summary_results["Better Bilevel"].append(
        int( 
            ((df_aggregated[f"{method}_feasible"]) & ( (~df_aggregated["bilevelsolver_feasible"]) | (df_aggregated[f"{method}_bestobj"] <= df_aggregated["bilevelsolver_bestobj"]-0.1) ) ).sum()  
        ) if method != "bilevelsolver" else np.nan
    )

    summary_results["Better Mibs"].append(
        int( 
            ((df_aggregated[f"{method}_feasible"]) & ( (~df_aggregated["mibs_feasible"]) | (df_aggregated[f"{method}_bestobj"] <= df_aggregated["mibs_bestobj"]-0.1) ) ).sum()  
        ) if method != "mibs" else np.nan
    )

    summary_results["Better Bilevel+DD"].append(
        int( 
            ((df_aggregated[f"{method}_feasible"]) & ( (~df_aggregated["bilevelsolver_plus_dd_feasible"]) | (df_aggregated[f"{method}_bestobj"] <= df_aggregated["bilevelsolver_plus_dd_bestobj"]-0.1) ) ).sum()  
        ) if method != "bilevelsolver_plus_dd" else np.nan
    )

    summary_results["Better Mibs+DD"].append(
        int( 
            ((df_aggregated[f"{method}_feasible"]) & ( (~df_aggregated["mibs_plus_dd_feasible"]) | (df_aggregated[f"{method}_bestobj"] <= df_aggregated["mibs_plus_dd_bestobj"]-0.1) ) ).sum()  
        ) if method != "mibs_plus_dd" else np.nan
    )

    # # Better than all others
    # others = [m for m in ["dd", "bilevelsolver", "mibs"] if m != method]
    # cond = df_aggregated[f"{method}_feasible"] & df_aggregated[f"{method}_improved_known_bobilib"]
    # for o in others:
    #     cond = cond & ( (~df_aggregated[f"{o}_feasible"]) |
    #                     (df_aggregated[f"{method}_bestobj"] < df_aggregated[f"{o}_bestobj"]) )
    # summary_results["Better All"].append(int(cond.sum()))

df_summary = pd.DataFrame(summary_results)
# df_summary["Better Bilevel"]
# print("\nSummary of Results:")
df_summary

In [ ]:
import pandas as pd
import numpy as np

# ---------- Settings ----------
use_booktabs = True
caption = "Summary of solver results on the BOBILib testbed."
label   = "tab:summary_results"

# Row order and LaTeX names
method_order = ['mibs', 'mibs_plus_dd', 'bilevelsolver', 'bilevelsolver_plus_dd', 'dd']
method_latex = {'mibs': r'$\MibS{}$', 'mibs_plus_dd': r'$\MibS+\DD{}$', 'bilevelsolver': r'$\BC{}$', 'bilevelsolver_plus_dd': r'$\BC+\DD{}$', 'dd': r'$\DD{}$'}

# Pairwise header strings (match your table)
col_lt_mibs = r"$\;\;< z_{\MibS{}}\;\;$"
col_lt_mibsdd = r"$\;\;< z_{\MibS{}+\DD{}}\;\;$"
col_lt_bc   = r"$\;\;< z_{\BC{}}\;\;$"
col_lt_bcdd = r"$\;\;< z_{\BC+\DD{}}\;\;$"
col_lt_dd   = r"$\;\;< z_{\DD{}}\;\;$"

dfT = df_summary.copy()

# ----- Assemble output with exact headers -----
df_out = pd.DataFrame({
    "Method": dfT["Method"],
    "Feasible": dfT["Has Feasible Solution"],
    "Optimal": dfT["Proved Optimality"],
    r"% Gap $\le$ 1\%": dfT["% Gap <= 1%"],
    r"% Gap $\le$ 10\%": dfT["% Gap <= 10%"],
    r"% Gap $\le$ 100\%": dfT["% Gap <= 100%"],
    col_lt_mibs: dfT["Better Mibs"],
    col_lt_mibsdd: dfT["Better Mibs+DD"],
    col_lt_bc:   dfT["Better Bilevel"],
    col_lt_bcdd: dfT["Better Bilevel+DD"],
    col_lt_dd:   dfT["Better DD"],
})

# Map method names and enforce order MibS -> Mibs+DD -> BC -> BC+DD -> DD
df_out["Method"] = df_out["Method"].map(method_latex).fillna(df_out["Method"])
order_key = df_summary["Method"].map({k: i for i, k in enumerate(method_order)})
if order_key.isna().all():  # fallback if df_summary already mapped
    order_key = df_out["Method"].map({method_latex[k]: i for i, k in enumerate(method_order)})
df_out = df_out.assign(_ord=order_key).sort_values("_ord").drop(columns="_ord").reset_index(drop=True)

# Format integers with thousands separators for counts and pairwise wins
count_cols = ["Feasible", "Optimal", col_lt_mibs, col_lt_mibsdd, col_lt_bc, col_lt_bcdd, col_lt_dd]
for c in count_cols:
    df_out[c] = df_out[c].apply(lambda x: f"{int(x):,}" if pd.notna(x) else "--")

# Format percentage columns with one decimal and a percent sign
pct_cols = [r"% Gap $\le$ 1\%", r"% Gap $\le$ 10\%", r"% Gap $\le$ 100\%"]
for c in pct_cols:
    df_out[c] = df_out[c].apply(lambda x: f"{float(x):.1f}\\%" if pd.notna(x) else "--")

# ----- Build LaTeX -----
cols = ["Method", "Feasible", "Optimal",
        r"% Gap $\le$ 1\%", r"% Gap $\le$ 10\%", r"% Gap $\le$ 100\%",
        col_lt_mibs, col_lt_mibsdd, col_lt_bc, col_lt_bcdd, col_lt_dd]

align_spec = "l" + "r"*(len(cols)-1)
if use_booktabs:
    head = (f"\\begin{{table}}[h!]\n\\centering\n\\small\n\\caption{{{caption}}}\n"
            f"\\label{{{label}}}\n\\begin{{tabular}}{{{align_spec}}}\n\\toprule\n")
    mid  = "\\midrule\n"
    foot = "\\bottomrule\n\\end{tabular}\n\\end{table}\n"
else:
    head = (f"\\begin{{table}}[h!]\n\\centering\n\\small\n\\caption{{{caption}}}\n"
            f"\\label{{{label}}}\n\\begin{{tabular}}{{{align_spec}}}\n\\hline\n")
    mid  = "\\hline\n"
    foot = "\\hline\n\\end{tabular}\n\\end{table}\n"

latex = head
latex += " {} \\\\\n".format(" & ".join(cols))
latex += mid
for _, row in df_out.iterrows():
    latex += " {} \\\\\n".format(" & ".join(str(row[c]) for c in cols))
latex += foot

print(latex)




MibS Analysis

In [ ]:
# Collect instances that are feasible for both, but not solved by either
df_plot = df_aggregated[
    (df_aggregated["mibs_feasible"] == 1)
    & (df_aggregated["mibs_plus_dd_feasible"] == 1)
    & (~df_aggregated["mibs_solved"])
    & (~df_aggregated["mibs_plus_dd_solved"])
].copy()

# Convert opt gap to percentages
df_plot["mibs_optgap_pct"] = 100 * df_plot["mibs_optgap"]
df_plot["mibs_plus_dd_optgap_pct"] = 100 * df_plot["mibs_plus_dd_optgap"]

# Long format for plotting
df_plot_melted = df_plot.melt(
    id_vars=["name"],
    value_vars=["mibs_optgap_pct", "mibs_plus_dd_optgap_pct"],
    var_name="method",
    value_name="opt_gap_pct",
)

df_plot_melted["method"] = df_plot_melted["method"].map({
    "mibs_optgap_pct": "MibS",
    "mibs_plus_dd_optgap_pct": "MibS + DD",
})

palette = {
    "MibS": colors["mibs"],
    "MibS + DD": colors["dd"],
}

plt.figure(figsize=(6, 4))
ax = sns.boxplot(
    data=df_plot_melted,
    x="method",
    y="opt_gap_pct",
    order=["MibS", "MibS + DD"],
    palette=palette,
    width=0.5,
    hue="method",
    legend=False,
)

plt.xlabel("Method", fontsize=fontsize)
plt.ylabel("Optimality Gap (%)", fontsize=fontsize)
plt.xticks(fontsize=fontsize)
plt.yticks(fontsize=fontsize)
plt.yscale("log")

plt.grid(True, color='lightgrey', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.savefig("figures/mibs_vs_mibsdd_optgap.pdf", format="pdf", bbox_inches="tight")
plt.show()


In [ ]:
df_plot[["mibs_optgap_pct", "mibs_plus_dd_optgap_pct"]].describe()

In [ ]:
df_plot[["mibs_optgap_pct", "mibs_plus_dd_optgap_pct"]].describe()

BilevelSolver Analysis

In [ ]:
# Collect instances that are feasible for both, but not solved by either
df_bilevel_plot = df_aggregated[
    (df_aggregated["bilevelsolver_feasible"] == 1)
    & (df_aggregated["bilevelsolver_plus_dd_feasible"] == 1)
    & (~df_aggregated["bilevelsolver_solved"])
    & (~df_aggregated["bilevelsolver_plus_dd_solved"])
    & (df_aggregated["bilevelsolver_optgap"] <= 1)
    & (df_aggregated["bilevelsolver_plus_dd_optgap"] <= 1)
].copy()

# Convert opt gap to percentages
df_bilevel_plot["bilevelsolver_optgap_pct"] = 100 * df_bilevel_plot["bilevelsolver_optgap"]
df_bilevel_plot["bilevelsolver_plus_dd_optgap_pct"] = 100 * df_bilevel_plot["bilevelsolver_plus_dd_optgap"]

# Long format for plotting
df_bilevel_plot_melted = df_bilevel_plot.melt(
    id_vars=["name"],
    value_vars=["bilevelsolver_optgap_pct", "bilevelsolver_plus_dd_optgap_pct"],
    var_name="method",
    value_name="opt_gap_pct",
)

df_bilevel_plot_melted["method"] = df_bilevel_plot_melted["method"].map({
    "bilevelsolver_optgap_pct": "BC",
    "bilevelsolver_plus_dd_optgap_pct": "BC + DD",
})

bilevel_palette = {
    "BC": colors["bilevelsolver"],
    "BC + DD": colors["dd"],
}

plt.figure(figsize=(6, 4))
ax = sns.boxplot(
    data=df_bilevel_plot_melted,
    x="method",
    y="opt_gap_pct",
    order=["BC", "BC + DD"],
    palette=bilevel_palette,
    width=0.5,
    hue="method",
    legend=False,
)

plt.xlabel("Method", fontsize=fontsize)
plt.ylabel("Optimality Gap (%)", fontsize=fontsize)
plt.xticks(fontsize=fontsize)
plt.yticks(fontsize=fontsize)
# plt.yscale("log")

plt.grid(True, color='lightgrey', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.savefig("figures/bc_vs_bcdd_optgap.pdf", format="pdf", bbox_inches="tight")
plt.show()


In [ ]:
df_bilevel_plot[["bilevelsolver_optgap_pct", "bilevelsolver_plus_dd_optgap_pct"]].describe()


In [ ]:
df_aggregated["bilevelsolver_feasible"].sum()

In [ ]:
df_aggregated["bilevelsolver_plus_dd_feasible"].sum()

In [ ]:
df_augmented[(df_augmented["solver"] == "bilevelsolver_plus_dd") & (df_augmented["width"] == based_width_augmented)].shape[0]

In [ ]:
df_augmented[(df_augmented["solver"] == "mibs_plus_dd") & (df_augmented["width"] == based_width_augmented)].shape[0]

In [ ]:
df[(df["solver"] == "mibs")].shape[0]

In [ ]:
df_aug = df_augmented[(df_augmented["solver"] == "mibs_plus_dd") & (df_augmented["width"] == based_width_augmented)].shape[0]
df_mibs = df[(df["solver"] == "mibs")]

In [ ]:
# find instances that are in df_mibs but not in df_aug
df_mibs_not_aug = df_mibs[(~df_mibs["name"].isin(df_augmented["name"])) & (df_mibs["solved"])]
df_mibs_not_aug.sort_values(by="state_variables")